In [1]:
kaggle = {"username":"taqiyudinadn","key":"4d6235e7fa7f0b8e84d9792db067b696"}

In [2]:
import json
import os
import subprocess
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from catboost import CatBoostRegressor, Pool
import pynvml

pd.set_option("display.max_columns", 200)

COMPETITION = "deescuy"
RANDOM_STATE = 42
DATA_DIR = Path("data") / COMPETITION
SUBMISSION_DIR = Path("submissions")
DATA_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# Bobot turnamen berdasarkan prestise.
TOURNAMENT_WEIGHT_RULES = {
    "fifa world cup": 2.00,
    "afc championship": 1.80,
    "afc asian cup": 1.90,
    "friendly": 0.96,
}
DEFAULT_TOURNAMENT_WEIGHT = 1.20


def get_tournament_weight(tournament_name: str) -> float:
    name = str(tournament_name).strip().lower()
    for key, weight in TOURNAMENT_WEIGHT_RULES.items():
        if key in name:
            return weight
    return DEFAULT_TOURNAMENT_WEIGHT


def compute_aw_mae(
    y_true_team,
    y_true_opp,
    y_pred_team,
    y_pred_opp,
    tournament_names,
    exact_weight: float = 0.30,
    outcome_weight: float = 0.25,
    gd_weight: float = 0.15,
    outcome_multiplier: float = 1.5,
    error_power: float = 1.5,
):
    """
    AW-MAE sesuai deskripsi kompetisi:
    1) MAE dasar dua target gol
    2) Bonus/Penalti exact, outcome, dan goal difference
    3) Jika outcome salah -> error x 1.5
    4) Error dipangkatkan 1.5
    5) Diakumulasi dengan bobot turnamen
    """
    y_true_team = np.asarray(y_true_team, dtype=float)
    y_true_opp = np.asarray(y_true_opp, dtype=float)
    y_pred_team = np.asarray(y_pred_team, dtype=float)
    y_pred_opp = np.asarray(y_pred_opp, dtype=float)

    y_true_team_int = np.rint(np.clip(y_true_team, 0, None)).astype(int)
    y_true_opp_int = np.rint(np.clip(y_true_opp, 0, None)).astype(int)
    y_pred_team_int = np.rint(np.clip(y_pred_team, 0, None)).astype(int)
    y_pred_opp_int = np.rint(np.clip(y_pred_opp, 0, None)).astype(int)

    base_mae = (np.abs(y_true_team - y_pred_team) + np.abs(y_true_opp - y_pred_opp)) / 2.0

    exact_match = (y_true_team_int == y_pred_team_int) & (y_true_opp_int == y_pred_opp_int)

    outcome_true = np.sign(y_true_team_int - y_true_opp_int)
    outcome_pred = np.sign(y_pred_team_int - y_pred_opp_int)
    outcome_match = outcome_true == outcome_pred

    gd_true = y_true_team_int - y_true_opp_int
    gd_pred = y_pred_team_int - y_pred_opp_int
    gd_match = gd_true == gd_pred

    # Bonus jika benar, penalti jika salah. Error dijaga tetap non-negatif.
    aug_error = base_mae.copy()
    aug_error += np.where(exact_match, -exact_weight, exact_weight)
    aug_error += np.where(outcome_match, -outcome_weight, outcome_weight)
    aug_error += np.where(gd_match, -gd_weight, gd_weight)
    aug_error = np.clip(aug_error, 0.0, None)

    aug_error = np.where(~outcome_match, aug_error * outcome_multiplier, aug_error)
    loss = np.power(aug_error, error_power)

    weights = np.asarray([get_tournament_weight(t) for t in tournament_names], dtype=float)
    weighted_score = float(np.sum(loss * weights) / np.sum(weights))

    detail = {
        "base_mae_mean": float(np.mean(base_mae)),
        "aug_error_mean": float(np.mean(aug_error)),
        "exact_match_rate": float(np.mean(exact_match)),
        "outcome_match_rate": float(np.mean(outcome_match)),
        "gd_match_rate": float(np.mean(gd_match)),
        "mean_tournament_weight": float(np.mean(weights)),
    }
    return weighted_score, detail

/tmp/ipykernel_12995/883163176.py:16: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


In [3]:
def ensure_kaggle_credentials():
    """
    Prioritas:
    1) ~/.kaggle/kaggle.json
    2) variabel `kaggle` di notebook
    3) env var KAGGLE_USERNAME + KAGGLE_KEY
    """
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"

    if kaggle_json.exists():
        return kaggle_json

    username = None
    key = None

    # Fallback ke dict kredensial yang disimpan di notebook.
    if "kaggle" in globals() and isinstance(globals()["kaggle"], dict):
        username = globals()["kaggle"].get("username")
        key = globals()["kaggle"].get("key")

    if not username or not key:
        username = os.getenv("KAGGLE_USERNAME")
        key = os.getenv("KAGGLE_KEY")

    if not username or not key:
        raise RuntimeError(
            "Kredensial Kaggle tidak ditemukan. "
            "Isi variabel `kaggle` di notebook, atau siapkan ~/.kaggle/kaggle.json, "
            "atau set KAGGLE_USERNAME dan KAGGLE_KEY."
        )

    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
    try:
        os.chmod(kaggle_json, 0o600)
    except Exception:
        pass
    return kaggle_json


def download_competition_data(competition: str, output_dir: Path):
    ensure_kaggle_credentials()

    csv_files = list(output_dir.rglob("*.csv"))
    if csv_files:
        print(f"Dataset sudah ada di {output_dir}. Lewati download.")
        return

    cmd = [
        "kaggle",
        "competitions",
        "download",
        "-c",
        competition,
        "-p",
        str(output_dir),
    ]
    print("Menjalankan:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    zip_files = list(output_dir.glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError("File zip hasil download tidak ditemukan.")

    for zf in zip_files:
        print(f"Extract: {zf.name}")
        with zipfile.ZipFile(zf, "r") as zip_ref:
            zip_ref.extractall(output_dir)

    print("Download + extract selesai.")


download_competition_data(COMPETITION, DATA_DIR)
print("Isi folder data:")
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(DATA_DIR))

Menjalankan: kaggle competitions download -c deescuy -p data/deescuy


 54%|█████▍    | 4.00M/7.37M [00:00<00:00, 8.70MB/s]


Extract: deescuy.zip
Download + extract selesai.
Isi folder data:
- deescuy.zip
- metadata.txt
- sample submission.csv
- test.csv
- train.csv


100%|██████████| 7.37M/7.37M [00:00<00:00, 10.4MB/s]
